**Group A**

*Jesús Sanz Alonso*

*Agustín Prieto Páez*

In [1]:
class State:
    def __init__(self,state,longitude,latitude):
        self.state = state
        self.longitude = longitude
        self.latitude = latitude

In [2]:
class Action: 
    def __init__(self,origin,destination,cost):
        self.origin = origin
        self.destination = destination
        self.cost = cost
    def __str__(self):
       return(
           f' {self.origin} → {self.destination} ({self.cost})'
       )

In [3]:
class Node:
    def __init__(self,parent,state,action,depth,accumulatedCost):
        self.parent = parent
        self.state = state # we want it to be type State
        self.action = action # the same with action is going to be a class
        self.depth = depth
        self.accumulatedCost = accumulatedCost # this way it's easier to compute g(n) on f(n) = g(n)+h(n)
        self.momento = 0
    def __str__(self):
        stringToReturn = (
        f'parent --> { self.parent}\n'
        f'state --> { self.state.state}\n'
        f'action --> (origin, destination,cost) --> ({self.action.origin} , {self.action.destination}, {self.action.cost})\n'
        f'depth -->  {self.depth}'
        )
        return stringToReturn
    def __lt__(self,obj):
        return self.momento < obj.momento

In [4]:
from collections import deque
class Search: # this is where we use inheritance
    def __init__(self,problem):
        self.openDS = deque()
        self.problem = problem
    def insert(self,successor):
        self.openDS.append(successor)
    def extract():
        pass

In [5]:
import statistics
import json
from collections import deque
class Problem:
    def __init__(self,file_name):#O(n)
        # Here, I read the dictionary
        self.nodesGenerated = 0
        self.exploredNodes = 0
        self.expandedNodes = 0
        self.depth = 0
        self.totalCost = 0.0
        self.explored = set()
        with open(file_name,'r') as file:
            self.dictionary = json.load(file)
            # Conversión de velocidad de km/h a m/s y cálculo del coste
       # Convert the list of intersections to a dictionary of dictionaries
        self.dictionary['maxSpeedOfAllSpeeds'] = float('-inf') # definiendo la maxima velocida a menos infinito
        self.dictionary['mostRepeatedSpeed'] = []
        self.dictionary['intersections'] = {inter['identifier']: inter for inter in self.dictionary.get('intersections')}# O(m)
        
        # Add the 'whereto' attribute to each intersection
        for inter in self.dictionary['intersections'].values():# O(m)
            inter['whereto'] = []

        # Populate the 'whereto' attribute based on the segments
        for segment in self.dictionary.get('segments'):# O(n)
            origin = segment['origin']#O(1)
            destination = segment['destination'] #O(1)
            distance = segment['distance'] #O(1)
            speed_kmh = segment['speed'] # O(1)

            # Convert speed from km/h to m/s
            speed_ms = speed_kmh * (1000 / 3600)
            self.dictionary['mostRepeatedSpeed'].append(speed_ms)
            # Si tenemos una velocidad mayor a la predeterminada, la cogemos
            if(speed_ms > self.dictionary.get('maxSpeedOfAllSpeeds')):#O(1)
                self.dictionary['maxSpeedOfAllSpeeds'] = speed_ms
    
            # Calculate the cost
            cost = distance / speed_ms
    
            # Add the destination and cost to the 'whereto' attribute of the origin intersection
            if origin in self.dictionary.get('intersections'): # O(1)
                self.dictionary.get('intersections').get(origin).get('whereto').append({'id': destination, 'cost': cost})
                # Convertir la lista de intersecciones a un diccionario donde la clave sea el 'identifier'
                #self.dictionary['intersections'] = {intersection['identifier']: {**intersection, 'whereto': set()} for intersection in self.dictionary.get('intersections')}
        self.dictionary['mostRepeatedSpeed'] = statistics.multimode(self.dictionary.get('mostRepeatedSpeed')) # O(1)
        self.initializeOpen(self.dictionary.get('initial')) # inicializo nodo raiz # O(1)
        
    def initializeOpen(self,initial): # O(1)
        longitudeInitialNode = self.dictionary.get('intersections').get(initial).get('longitude')
        latitudeInitialNode = self.dictionary.get('intersections').get(initial).get('latitude')
        self.root = Node(None,State(initial,longitudeInitialNode,latitudeInitialNode),Action(None,initial,0),0,0) # no estoy seguro si para llegar al nodo raiz action == None
        self.nodesGenerated+=1
        self.root.momento = self.nodesGenerated
    #################################################################################
    ####################             search              ############################
    #################################################################################
    def search(self,search_param): # O(n)
        """:param search_param: strategy to use
        
        :returns: empty list of list of actions"""
        search_param.insert(self.root)
        while len(search_param.openDS)!=0:    
            node = search_param.extract() # O(1) 
            self.exploredNodes +=1
            if node.state.state not in self.explored:
                if(self.testGoal(node)): 
                    self.depth = node.depth
                    self.totalCost = node.accumulatedCost
                    return self.recoverPath(node,[],0)
                successors1 = self.expand(node) # O(n)
                if (len(successors1)>0):
                    self.expandedNodes+=1
                for  successor in successors1: # O(n)
                    search_param.insert(successor) # O(1)
                self.explored.add(node.state.state) #  node.state es el objeto y node.state.state es la variable en el objeto state
        print("Solución no encontrada y hemos recorrido todo el árbol")
        return search_param.openDS

     #################################################################################
    ####################             testGoal              ############################
    #################################################################################
    def testGoal(self,node):# O(1)
        return self.dictionary.get('final') == node.state.state# node.state es de tipo State y node.state.state es de tipo int
    #################################################################################
    ####################             expand             ############################
    #################################################################################
    def expand(self,Node_param): #O(n)
        """ 
        :param Node_param: nodo al que apuntamos 
        :returns: list of nodes """
        successors = []
        currentIntersection = self.dictionary.get('intersections').get(Node_param.state.state) # O(1)

        listOrdered = sorted(currentIntersection.get("whereto"), key = lambda x:x['id']) # O(n*log(n)) # Timsort
        for destination in listOrdered: # O(n)
            """currentIntersection.get("whereto")
            [{'id': 1256026663, 'cost': 1.7331}, {'id': 1531659796, 'cost': 2.346}]"""
            if destination.get('id') in self.explored: # O(1) # preguntamos si ya lo hemos recorrido
                continue
            newAction = Action(# O(1)
                    Node_param.state.state, #origen
                    destination.get("id"), # destino
                    destination.get("cost") #coste
                )
            # REMEMBER THAT destination is A DICTIONARY {"id":,"cost":}
            newState = State(newAction.destination,self.dictionary.get('intersections').get(destination.get('id')).get('longitude'),self.dictionary.get('intersections').get(destination.get('id')).get('latitude')) #self.applyAction(Node_param.state,action) # Node.state es un objeto de tipo state
            newNode = Node(Node_param,newState,newAction,Node_param.depth+1,Node_param.accumulatedCost+newAction.cost)
            self.nodesGenerated+=1
            newNode.momento = self.nodesGenerated
            successors.append(newNode)
        return successors
    def recoverPath(self,node,list_param,total_cost):# O(n)
        if node.parent is None:# O(1)
            temp = []# O(1)
            i = deque(list_param)
            while  len(i) != 0:# O(n)
                temp.append(i.pop())# O(1)
            return temp
        else:
            list_param.append(node.action)# O(1)
            total_cost = total_cost + node.action.cost
            return self.recoverPath(node.parent,list_param=list_param,total_cost=total_cost) # O(T) # not that expensive it could be worse

In [6]:
import heapq
from geographiclib.geodesic import Geodesic # pip install geographiclib
class InformedSearch(Search):
    """notice that we work with a tuple
    so if we want to return a node we must say tuple[1]
    where the tuple is (heuristic,node)"""
    def extract(self): #O(1)
        # tenemos que extraer el que menor heurística tiene --> la heurística hace de prioridad
        return heapq.heappop(self.openDS)[1] # heapq es una priorityQueue sin ser una Queue sino una Heap
    def computeHeuristic(self,node_param): #O(1)
        """self.openDS is by default a deque() (see Search __init__) so we
        have to convert deque() into a list"""
        self.openDS = list(self.openDS) # medios para obtener lo que queremos
        goalId = self.problem.dictionary.get('final') # Obtenemos estado final
        coord_1 = (node_param.state.longitude, node_param.state.latitude) # (longitude,latitude)
        coord_2 = (self.problem.dictionary.get('intersections').get(goalId).get('longitude'), 
                   self.problem.dictionary.get('intersections').get(goalId).get('latitude'))  # (longitude,latitude)
        # Usar el elipsoide WGS84 para calcular la distancia
        geod = Geodesic.WGS84 # cosas de la librería
        resultado = geod.Inverse(coord_1[0], coord_1[1], coord_2[0], coord_2[1]) # cosas de la librería
        # Distancia en metros
        distancia = resultado['s12'] 
        return distancia

In [7]:
from collections import deque
class DepthFirst(Search): # LIFO queue
    def extract(self):#O(1)
        return self.openDS.pop() 

In [8]:
class BreadthFirst(Search):# FIFO queue    
    def extract(self): # O(1)
        return self.openDS.popleft() # extrae por la izquierda (el primero en llegar)

In [9]:
import heapq # O(log n )
class BestFirst(InformedSearch): # takes into account only h(n)
     # element is a node
     def insert(self,element): #O(1)
        """self.openDS is by default a deque() (see Search __init__) so we
        have to convert deque() into a list"""
        self.openDS = list(self.openDS)
        heuristic = super().computeHeuristic(element)
        #print('\n-----------\n'.join(map(str,self.openDS)))
        heapq.heappush(self.openDS,(heuristic,element)) # element is going to be a paired value (h,Node)

In [10]:
import heapq
class AStarOptimisticButRealistic(InformedSearch):
    def insert(self,element): #O(1)
         # element is a node
        """self.openDS is by default a deque() (see Search __init__) so we
        have to convert deque() into a list"""
        self.openDS = list(self.openDS)
        # f(n) = h(n) + g(n) donde la heuristica es la distancia euclidea a la meta entre la maxima velocidad de entre todas las velocidades que tenemos  
        heuristic = (super().computeHeuristic(element)/self.problem.dictionary.get('maxSpeedOfAllSpeeds'))+element.accumulatedCost
        #print('\n-----------\n'.join(map(str,self.openDS)))
        heapq.heappush(self.openDS,(heuristic,element)) # element is going to be a paired value (h,Node)

In [11]:
import heapq
class AStarGeodesicWithMostRepeatedSpeed(InformedSearch): #O(1)
    def insert(self,element):
        distancia = super().computeHeuristic(element) 
        # f(n) = h(n) + g(n) donde la heuristica es la distancia euclidea a la meta entre la velocidad más repetida entre todas las velocidades que tenemos  
        heuristic = (distancia/self.problem.dictionary.get('mostRepeatedSpeed')[0])+element.accumulatedCost
        #print('\n-----------\n'.join(map(str,self.openDS)))
        heapq.heappush(self.openDS,(heuristic,element)) # element is going to be a paired value (h,Node)

In [12]:
import heapq
class AStarAssumingOneHundredAndTwentyKilometersPerHour(InformedSearch):
    def insert(self,element):
         # element is a node
        """self.openDS is by default a deque() (see Search __init__) so we
        have to convert deque() into a list"""
        self.openDS = list(self.openDS)
        hundredTwentyKilometersPerHour = 120
        metersPerSecond = hundredTwentyKilometersPerHour * (1000/3600)
        # f(n) = h(n) + g(n) donde la heuristica es la distancia euclidea a la meta entre la maxima velocidad de entre todas las velocidades que tenemos  
        heuristic = (super().computeHeuristic(element)/metersPerSecond)+element.accumulatedCost
        #print('\n-----------\n'.join(map(str,self.openDS)))
        heapq.heappush(self.openDS,(heuristic,element)) # element is going to be a paired value (h,Node)

In [13]:
import heapq
class AStar(InformedSearch):# takes into account g(n), not only h(n)
    def insert(self,element):
         # element is a node
        """self.openDS is by default a deque() (see Search __init__) so we
        have to convert deque() into a list"""
        self.openDS = list(self.openDS)
        heuristic = (super().computeHeuristic(element)/self.problem.dictionary.get('maxSpeedOfAllSpeeds'))+element.accumulatedCost
        #print('\n-----------\n'.join(map(str,self.openDS)))
        heapq.heappush(self.openDS,(heuristic,element)) # element is going to be a paired value (h,Node)

In [26]:
import time
import os
class Main:
    def main(self):
        for j in ['huge','large','medium','small']:
            directorio = "C:\\googleMapsVS\\Google-Maps\\SUBMISSION\\examples_with_solutions\\problems\\"+j #TODO: CHANGE PATH
            archivos = os.listdir(directorio)
            print(f"#################################################")
            print(f"#                    COMENZAMOS                 #")
            print(f"#################################################")

            for i in archivos:
                os.chdir(directorio)
                problem = Problem(i)
                start = time.time()
                busqueda = AStar(problem)
                print(f"#################################################")
                print(f"#                       {busqueda.__class__.__name__}                      #")
                print(f"#################################################")
                print(f"#                       {i}                     #")
                print(f"#################################################")
                result = problem.search(busqueda)
                end = time.time()
                print(f'Generated nodes: {problem.nodesGenerated}\n')
                print(f'Expanded nodes: {problem.expandedNodes}\n')
                print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                print(f'Solution length: {problem.depth}\n')
                print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                print(f'Solution: [')
                print(','.join(map(str,result)))
                print(f']')
                ############################################################################
                os.chdir(directorio)
                problem = Problem(i)
                start = time.time()
                busqueda = BreadthFirst(problem)
                print(f"#################################################")
                print(f"#                       {busqueda.__class__.__name__}                      #")
                print(f"#################################################")
                print(f"#                       {i}                     #")
                print(f"#################################################")
                result = problem.search(busqueda)
                end = time.time()
                print(f'Generated nodes: {problem.nodesGenerated}\n')
                print(f'Expanded nodes: {problem.expandedNodes}\n')
                print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                print(f'Solution length: {problem.depth}\n')
                print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                print(f'Solution: [')
                print(','.join(map(str,result)))
                print(f']')
                ############################################################################
                os.chdir(directorio)
                problem = Problem(i)
                start = time.time()
                busqueda = DepthFirst(problem)
                print(f"#################################################")
                print(f"#                       {busqueda.__class__.__name__}                      #")
                print(f"#################################################")
                print(f"#                       {i}                     #")
                print(f"#################################################")
                result = problem.search(busqueda)
                end = time.time()
                print(f'Generated nodes: {problem.nodesGenerated}\n')
                print(f'Expanded nodes: {problem.expandedNodes}\n')
                print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                print(f'Solution length: {problem.depth}\n')
                print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                print(f'Solution: [')
                print(','.join(map(str,result)))
                print(f']')
                #############################################################################
                os.chdir(directorio)
                problem = Problem(i)
                start = time.time()
                busqueda = BestFirst(problem)
                print(f"#################################################")
                print(f"#                       {busqueda.__class__.__name__}                      #")
                print(f"#################################################")
                print(f"#                       {i}                     #")
                print(f"#################################################")
                result = problem.search(busqueda)
                end = time.time()
                print(f'Generated nodes: {problem.nodesGenerated}\n')
                print(f'Expanded nodes: {problem.expandedNodes}\n')
                print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                print(f'Solution length: {problem.depth}\n')
                print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                print(f'Solution: [')
                print(','.join(map(str,result)))
                print(f']')
                # Ctrl + / para descomentar <--
                # #############################################################################
                # os.chdir(directorio)
                # problem = Problem(i)
                # start = time.perf_counter()
                # busqueda = AStarOptimisticButRealistic(problem)
                # print(f"#################################################")
                # print(f"#                       {busqueda.__class__.__name__}                      #")
                # print(f"#################################################")
                # print(f"#                       {i}                     #")
                # print(f"#################################################")
                # result = problem.search(busqueda)
                # end = time.perf_counter()
                # print(f'Generated nodes: {problem.nodesGenerated}\n')
                # print(f'Expanded nodes: {problem.expandedNodes}\n')
                # print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                # print(f'Solution length: {problem.depth}\n')
                # print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                # print(f'Solution: [')
                # print(','.join(map(str,result)))
                # print(f']')
                # #############################################################################
                # os.chdir(directorio)
                # problem = Problem(i)
                # start = time.perf_counter()
                # busqueda = AStarAssumingOneHundredAndTwentyKilometersPerHour(problem)
                # print(f"#################################################")
                # print(f"#                       {busqueda.__class__.__name__}                      #")
                # print(f"#################################################")
                # print(f"#                       {i}                     #")
                # print(f"#################################################")
                # result = problem.search(busqueda)
                # end = time.perf_counter()
                # print(f'Generated nodes: {problem.nodesGenerated}\n')
                # print(f'Expanded nodes: {problem.expandedNodes}\n')
                # print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                # print(f'Solution length: {problem.depth}\n')
                # print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                # print(f'Solution: [')
                # print(','.join(map(str,result)))
                # print(f']')
                # #############################################################################
                # os.chdir(directorio)
                # problem = Problem(i)
                # start = time.perf_counter()
                # busqueda = AStarGeodesicWithMostRepeatedSpeed(problem)
                # print(f"#################################################")
                # print(f"#                       {busqueda.__class__.__name__}                      #")
                # print(f"#################################################")
                # print(f"#                       {i}                     #")
                # print(f"#################################################")
                # result = problem.search(busqueda)
                # end = time.perf_counter()
                # print(f'Generated nodes: {problem.nodesGenerated}\n')
                # print(f'Expanded nodes: {problem.expandedNodes}\n')
                # print(f'Execution time: {self.formatear_segundos(end-start)}\n')
                # print(f'Solution length: {problem.depth}\n')
                # print(f'Solution cost: {self.formatear_segundos(problem.totalCost)}\n')
                # print(f'Solution: [')
                # print(','.join(map(str,result)))
                # print(f']')
    def formatear_segundos(self,segundos):
        horas = int(segundos // 3600)
        minutos = int((segundos % 3600) // 60)
        segundos_restantes = segundos % 60
        return f"{horas:02}:{minutos:02}:{segundos_restantes:02}"

In [27]:
hola = Main()
hola.main()

#################################################
#                    COMENZAMOS                 #
#################################################
#################################################
#                       AStar                      #
#################################################
#                       calle_agustina_aroca_albacete_5000_0.json                     #
#################################################
Generated nodes: 1023

Expanded nodes: 669

Execution time: 00:00:0.15624642372131348

Solution length: 55

Solution cost: 00:05:56.64627000000007

Solution: [
 1540051673 → 1990894635 (8.34984), 1990894635 → 1530764262 (12.91296), 1530764262 → 1526257441 (5.9064), 1526257441 → 1526221518 (5.44272), 1526221518 → 1529476077 (13.088399999999998), 1529476077 → 1529476057 (12.380399999999998), 1529476057 → 1529623355 (9.041999999999998), 1529623355 → 1529476050 (6.24864), 1529476050 → 1526221480 (2.3377199999999996), 1526221480 → 344738924 (4.43159999999999